In [6]:
import polars as pl
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import RobustScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline

In [7]:
df = pl.read_csv("../hospital_readmission_dataset.csv")
df.head()

patient_id,admission_date,season,age,gender,region,primary_diagnosis,comorbidities_count,length_of_stay,treatment_type,medications_count,followup_visits_last_year,prev_readmissions,insurance_type,discharge_disposition,readmission_risk_score,label
str,str,str,i64,str,str,str,i64,i64,str,i64,i64,i64,str,str,f64,i64
"""P00001""","""2022-04-14""","""Spring""",66,"""Male""","""South""","""Diabetes""",5,6,"""Interventional""",8,6,1,"""Medicare""","""Home Health""",0.92,1
"""P00002""","""2021-09-19""","""Fall""",55,"""Male""","""South""","""Diabetes""",4,6,"""Interventional""",6,4,3,"""Private""","""Home Health""",0.88,1
"""P00003""","""2023-04-12""","""Spring""",69,"""Female""","""West""","""Hypertension""",6,8,"""Medical""",9,6,2,"""Medicare""","""Skilled Nursing""",0.97,1
"""P00004""","""2023-08-14""","""Summer""",83,"""Male""","""South""","""Stroke""",6,11,"""Medical""",11,4,2,"""Medicare""","""Skilled Nursing""",0.97,1
"""P00005""","""2021-11-05""","""Fall""",54,"""Female""","""North""","""Stroke""",4,10,"""Medical""",6,2,1,"""Uninsured""","""Home Health""",0.83,1


In [8]:
df.describe()

statistic,patient_id,admission_date,season,age,gender,region,primary_diagnosis,comorbidities_count,length_of_stay,treatment_type,medications_count,followup_visits_last_year,prev_readmissions,insurance_type,discharge_disposition,readmission_risk_score,label
str,str,str,str,f64,str,str,str,f64,f64,str,f64,f64,f64,str,str,f64,f64
"""count""","""8000""","""8000""","""8000""",8000.0,"""8000""","""8000""","""8000""",8000.0,8000.0,"""8000""",8000.0,8000.0,8000.0,"""8000""","""8000""",8000.0,8000.0
"""null_count""","""0""","""0""","""0""",0.0,"""0""","""0""","""0""",0.0,0.0,"""0""",0.0,0.0,0.0,"""0""","""0""",0.0,0.0
"""mean""",null,null,null,57.411625,null,null,null,4.3185,7.789125,null,7.47575,3.638125,1.5725,null,null,0.7769375,0.772875
"""std""",null,null,null,16.724388,null,null,null,1.358046,1.930252,null,2.287917,1.630415,0.89196,null,null,0.219885,0.419
"""min""","""P00001""","""2021-01-01""","""Fall""",18.0,"""Female""","""Central""","""Appendicitis""",1.0,3.0,"""Conservative""",2.0,0.0,0.0,"""Medicaid""","""Home""",0.07,0.0
"""25%""",null,null,null,46.0,null,null,null,3.0,6.0,null,6.0,2.0,1.0,null,null,0.63,1.0
"""50%""",null,null,null,57.0,null,null,null,4.0,8.0,null,8.0,4.0,1.0,null,null,0.86,1.0
"""75%""",null,null,null,69.0,null,null,null,5.0,9.0,null,9.0,4.0,2.0,null,null,0.97,1.0
"""max""","""P08000""","""2023-12-31""","""Winter""",95.0,"""Male""","""West""","""Stroke""",10.0,15.0,"""Surgical""",18.0,10.0,5.0,"""Uninsured""","""Skilled Nursing""",0.97,1.0


In [9]:
X = df.drop(['label'])
y = df['label']

# Define column groups
numerical_cols = [
    'age', 'comorbidities_count', 'length_of_stay',
    'medications_count', 'followup_visits_last_year',
    'prev_readmissions', 'readmission_risk_score'
]
categorical_cols = [
    'season', 'gender', 'region', 'primary_diagnosis',
    'treatment_type', 'insurance_type', 'discharge_disposition'
]


In [10]:
# Split Test and Training data
# Convert polars DataFrame and Series to pandas DataFrame and Series
X_train, X_test, y_train, y_test = train_test_split(X.to_pandas(),
                                                    y.to_pandas(),
                                                    test_size=0.2,
                                                    random_state=42,
                                                    stratify=y)

In [12]:
# Build preprocessing pipeline
preprocessor = ColumnTransformer(transformers=[
    ('num', RobustScaler(), numerical_cols),
    ('cat', OneHotEncoder(drop='first', handle_unknown='ignore', sparse_output=False), categorical_cols)
])

# Fit on train only, transform both
X_train_scaled = preprocessor.fit_transform(X_train)
X_test_scaled  = preprocessor.transform(X_test)

In [16]:
pl.DataFrame(X_train_scaled).describe()

statistic,column_0,column_1,column_2,column_3,column_4,column_5,column_6,column_7,column_8,column_9,column_10,column_11,column_12,column_13,column_14,column_15,column_16,column_17,column_18,column_19,column_20,column_21,column_22,column_23,column_24,column_25,column_26,column_27,column_28,column_29,column_30,column_31,column_32,column_33
str,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64
"""count""",6400.0,6400.0,6400.0,6400.0,6400.0,6400.0,6400.0,6400.0,6400.0,6400.0,6400.0,6400.0,6400.0,6400.0,6400.0,6400.0,6400.0,6400.0,6400.0,6400.0,6400.0,6400.0,6400.0,6400.0,6400.0,6400.0,6400.0,6400.0,6400.0,6400.0,6400.0,6400.0,6400.0,6400.0
"""null_count""",0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
"""mean""",0.020136,0.16125,-0.066771,-0.172656,-0.183516,0.57375,-0.244729,0.249531,0.247656,0.253594,0.476094,0.198906,0.1878125,0.248125,0.206875,0.1196875,0.182969,0.054375,0.0509375,0.161094,0.028281,0.1096875,0.052969,0.06125,0.1115625,0.211094,0.4496875,0.245625,0.317969,0.381406,0.099375,0.754531,0.006406,0.19875
"""std""",0.732484,0.683461,0.643381,0.764821,0.81292,0.899479,0.649384,0.432776,0.431685,0.435102,0.499467,0.399209,0.390593,0.431958,0.405097,0.324621,0.386671,0.226774,0.219887,0.367646,0.165788,0.312524,0.223989,0.239807,0.314852,0.408117,0.497501,0.430491,0.465724,0.48577,0.299188,0.430398,0.079788,0.399091
"""min""",-1.695652,-1.5,-1.666667,-2.0,-2.0,-1.0,-2.323529,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
"""25%""",-0.478261,-0.5,-0.666667,-0.666667,-1.0,0.0,-0.676471,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0
"""50%""",0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0
"""75%""",0.521739,0.5,0.333333,0.333333,0.0,1.0,0.323529,0.0,0.0,1.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,1.0,1.0,0.0,1.0,0.0,0.0
"""max""",1.652174,3.0,2.333333,3.333333,3.0,4.0,0.323529,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0
